In [5]:
import carla

# Kết nối tới server CARLA
SERVER_HOST = 'localhost'  # Đổi thành IP server nếu notebook chạy ở máy khác
SERVER_PORT = 2000

client = carla.Client(SERVER_HOST, SERVER_PORT)
client.set_timeout(30.0)
world = client.get_world()
print('Đã kết nối CARLA tại {}:{} | Map: {}'.format(
    SERVER_HOST, SERVER_PORT, world.get_map().name
))

In [6]:
# Xem danh sách các bản đồ có sẵn
print(client.get_available_maps())


['/Game/Carla/Maps/Town01', '/Game/Carla/Maps/Town02', '/Game/Carla/Maps/Town03', '/Game/Carla/Maps/Town04', '/Game/Carla/Maps/Town05']


In [7]:
# Lệnh chuyển map01
client.load_world('Town01')

In [8]:
# Lệnh chuyển map02
client.load_world('Town02')

In [9]:
# Lệnh chuyển map03
client.load_world('Town03')

In [10]:
# Lệnh chuyển map04
client.load_world('Town04')

In [11]:
# Lệnh chuyển map05
client.load_world('Town05')

## Điều khiển thời tiết CARLA

Chạy cell tạo hàm một lần, sau đó chạy cell preset hoặc thời tiết tùy chỉnh. Các hàm luôn lấy world hiện tại nên vẫn dùng được sau khi đổi map.

In [ ]:
# Tự lấy toàn bộ preset từ đúng phiên bản CARLA đang chạy
WEATHER_PRESETS = {
    name: getattr(carla.WeatherParameters, name)
    for name in dir(carla.WeatherParameters)
    if name and name[0].isupper()
}

def set_weather_preset(name):
    if name not in WEATHER_PRESETS:
        available = ', '.join(sorted(WEATHER_PRESETS))
        raise ValueError('Preset không hợp lệ. Các preset hiện có: ' + available)
    world = client.get_world()
    world.set_weather(WEATHER_PRESETS[name])
    print('Đã đặt thời tiết:', name, '| Map:', world.get_map().name)
    return world.get_weather()

print('Preset thời tiết có thể dùng:')
print(', '.join(sorted(WEATHER_PRESETS)))

In [ ]:
# Đổi tên preset rồi chạy lại cell này
set_weather_preset('ClearNoon')

# Ví dụ khác:
# set_weather_preset('HardRainNoon')
# set_weather_preset('CloudySunset')

In [ ]:
# Hàm chỉnh chi tiết. Phần lớn giá trị nằm trong khoảng 0-100.
_WEATHER_RANGES = {
    'cloudiness': (0.0, 100.0),
    'precipitation': (0.0, 100.0),
    'precipitation_deposits': (0.0, 100.0),
    'wind_intensity': (0.0, 100.0),
    'fog_density': (0.0, 100.0),
    'fog_distance': (0.0, 1000.0),
    'wetness': (0.0, 100.0),
    'sun_azimuth_angle': (0.0, 360.0),
    'sun_altitude_angle': (-90.0, 90.0),
}

def set_custom_weather(**values):
    world = client.get_world()
    weather = world.get_weather()

    for key, value in values.items():
        if key not in _WEATHER_RANGES:
            raise ValueError('Tham số không hỗ trợ: ' + key)
        if not hasattr(weather, key):
            raise ValueError('CARLA hiện tại không có tham số: ' + key)
        low, high = _WEATHER_RANGES[key]
        value = float(value)
        if not low <= value <= high:
            raise ValueError('{} phải nằm trong khoảng {} đến {}'.format(key, low, high))
        setattr(weather, key, value)

    world.set_weather(weather)
    print('Đã cập nhật thời tiết tùy chỉnh | Map:', world.get_map().name)
    return world.get_weather()

In [ ]:
# Ví dụ: trời mưa, nhiều mây, đường ướt và hơi có sương
set_custom_weather(
    cloudiness=80,
    precipitation=60,
    precipitation_deposits=70,
    wind_intensity=35,
    fog_density=10,
    wetness=75,
    sun_azimuth_angle=30,
    sun_altitude_angle=25,
)

In [ ]:
# Đọc lại thời tiết hiện tại từ server
weather = client.get_world().get_weather()
for field in _WEATHER_RANGES:
    if hasattr(weather, field):
        print('{:<26} {}'.format(field + ':', getattr(weather, field)))